In [ ]:
import os
import sys
import argparse
from pathlib import Path, PurePath

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib import cbook

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
basin = 'animas'
dt = None
snowprop = 'depth'
reproj_type = 'uniformreproj'
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/spatial/diff_plots/depth_veg_topo'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
in_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'
verbose = True
overwrite = False
fallback_epsg = 'EPSG:32613'

In [ ]:
# Find and load topo.nc
topo_fn = h.fn_list(script_dir, f'{basin}_setup/output_100m/topo.nc', verbose=verbose)[0]
topo = h.load(topo_fn)
topo

In [ ]:
# Locate difference files
# defaults to all available dates in the basin if dt is None
if dt is not None:
    diff_fn_list = h.fn_list(in_dir, f'{basin}_wy*_diff_{dt}_{snowprop}_{reproj_type}.nc')
else:
    diff_fn_list = h.fn_list(in_dir, f'{basin}_wy*_diff_*_{snowprop}_{reproj_type}.nc')
if len(diff_fn_list) == 0:
    print(f'ERROR: No difference files found in {in_dir} for {basin}')
    sys.exit(1)

diff_fn_list

In [ ]:
for diff_fn in diff_fn_list:
    diff = h.load(diff_fn)
    ax = h.plot_one(diff, vmin=-1, vmax=1, cmap='RdYlBu', return_ax=True, figsize=(4,4), title=diff.description)
    ax.set_facecolor('k')

In [ ]:
# Extract the topo variables of interest
veg_height = topo['veg_height'].load()
veg_type = topo['veg_type'].load()
slope = topo['slope'].load()

In [ ]:
# Define vegetation classes
veg_height_classes = [0.00, 0.25, 0.75, 1.00, 2.00, 2.50, 3.00, 7.50, 17.50, 37.50]

# There are a bunch of these, don't process for now, need to aggregate better somehow
# veg_type_classes

# Make slope bins in increments of 10 degrees
slope_bins_arr = proc.bin_slope(slope, plot_on=False, basinname=basin)
slope_bins_arr

In [ ]:
# Now do a test boxplot of the first diff array in the diff list
diff = h.load(diff_fn_list[0])
ax = h.plot_one(diff, sd_switch=True, vmin=0, vmax=4, return_ax=True, figsize=(4,4), title=diff.description)
ax.set_facecolor('k')

In [ ]:
# Now group the diff arrays by slope bin, with handling for masked values and nans in the diff array
diff_mask = np.ma.masked_invalid(diff.values)
# Mask slope array by diff_mask
slope_masked = np.ma.masked_array(slope_bins_arr.values, mask=diff_mask.mask)
h.plot_one(slope_masked)

In [ ]:
# Extract boxplot stats from the difference array based on the input topo var classes
# Compute unique slope bins from the categorized array
slope_bins = np.unique(slope_bins_arr)

# Set up a list for this difference array, perhaps use the filename for when there are a bunch of them?
diff_slope_vals = []
# Loop through the slope bins
for slope_bin in slope_bins:
    # print(slope_bin)
    # Get the diff data for this slope bin that is unmasked
    diff_slope_bin = diff_mask[~slope_masked.mask & (slope_masked==slope_bin)]

    # Calculate boxplot statistics
    # Change the labels to slope intervals during plotting
    # Current label is the valid pixel count of difference values in each bin
    stats = cbook.boxplot_stats(diff_slope_bin, labels=[int(diff_slope_bin.size)])
    # Store this for plotting later
    diff_slope_vals.append(stats)

diff_slope_vals

In [ ]:
def group_vals(var_bins_arr, arr_fn=None, arr=None, v="slope", bins=None):

    # Load the array if input filename
    if arr_fn is not None:
        arr = h.load(arr_fn)
    # Otherwise the array needs to be input and arr cannot be None
    elif arr is not None:
        arr = arr
    else:
        print('ERROR: Must input either arr_fn or arr to group_vals function')
        return
    # Handle nans in the diff array
    arr_mask = np.ma.masked_invalid(arr.values)
    # Mask variable array using the arr_mask (this is smaller due to differencing with ASO)
    var_masked = np.ma.masked_array(var_bins_arr.values, mask=arr_mask.mask)

    # Extract boxplot stats from the difference array based on the input topo var classes
    # Compute unique bins from the categorized array if the variable is slope
    if v == 'slope':
        bins = np.unique(var_bins_arr)
    else:
        bins = bins

    # Set up a list for this difference array, perhaps use the filename for when there are a bunch of them?
    vals_grouped = []
    # Loop through the bins
    for thisbin in bins:
        # print(thisbin)
        # Get the diff data for this slope bin that is unmasked
        var_bin = arr_mask[~var_masked.mask & (var_masked==thisbin)]

        # Calculate boxplot statistics
        # Change the labels later on during plotting
        # Current label is the valid pixel count of difference values in each bin
        stats = cbook.boxplot_stats(var_bin, labels=[int(var_bin.size)])
        # Store stats for plotting
        vals_grouped.append(stats)

    return vals_grouped

In [ ]:
slope_bins_arr = proc.bin_slope(slope, plot_on=False, basinname=basin)
diff_vals_grouped = group_vals(arr_fn=diff_fn_list[0], var_bins_arr=slope_bins_arr)

In [ ]:
def plot_boxplot(vals, arr_fn=None, title=None, v='slope', ylims=(-0.5, 1.5),
                 med_xshift=-0.075, med_on=True, med_shift=0.2, units='m',
                 ylab='snow depth difference [m]', bin_labels=None, rot=False):
   # Plot the boxplot for this diff array
   fig, ax = plt.subplots()
   for jdx, s in enumerate(vals):
      ax.bxp(s, patch_artist=True,
            boxprops={'facecolor': 'navy'},
            showfliers=False,
         #    flierprops={'flierson': 'False'},
            positions=[jdx/3], widths=0.1)

   # Annotate the xtick labels of valid pixel counts for each category next to each boxplot
   for jdx, (s, xpos) in enumerate(zip(vals, ax.get_xticks())):
      pix_count = s[0]['label']
      if pix_count > 0:
         # mu = s[0]['mean']
         med = s[0]['med']
         # p75 = s[0]['q3']
         ax.text(xpos - 0.05, med, f'{pix_count}', ha='right', va='center', rotation=90, fontsize=8)
         if v != 'slope' and med == 0:
            continue
         else:
            if med_on:
               ax.text(xpos + med_xshift, med + med_shift, f'{med:.1f}{units}',
               ha='right', va='center', fontsize=8)
      else:
         continue

   ax.hlines(0, ax.get_xlim()[0], ax.get_xlim()[1], 'lightgray', '--')
   ax.set_ylim(ylims)
   ax.set_ylabel(ylab)
   ax.grid(visible=True, linestyle=':', color='grey', alpha=0.5)

   # Split out a title
   if arr_fn is not None:
      # # the basin, runtype, date, and projection type
      # title = f"{arr_fn.split('_')[0].capitalize()} {' '.join(arr_fn.split('_')[2:])}"
      # the basin, runtype and date
      title = f"{arr_fn.split('_')[0].capitalize()} {' '.join(arr_fn.split('_')[2:-1])}"
   else:
      title = title
   ax.set_title(f'{v.capitalize()} bins: {title}')

   # Create bin labels for the x-axis
   if v == 'slope':
      # slope labels in 10 degree intervals
      bin_labels = [f'{i}-{i+10}˚' for i in range(0, 60, 10)]
   else:
      bin_labels = bin_labels

   # Change the xtick labels to bin labels
   ax.set_xticklabels(bin_labels[:len(vals)], rotation=90 if rot else 0)

plot_boxplot(vals=diff_vals_grouped, arr_fn=PurePath(diff_fn_list[0]).stem)

In [ ]:
for diff_fn in diff_fn_list:
    # This works fine for iSnobal runs, needs handling for the original other types, try uniform reprojection now
    diff_vals_grouped = group_vals(arr_fn=diff_fn, var_bins_arr=slope_bins_arr)
    plot_boxplot(vals=diff_vals_grouped, arr_fn=PurePath(diff_fn).stem)

# Plot snow depth against slope

In [ ]:
snowprop = 'depth'
reproj_type = 'uniformreproj'
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/spatial/depth_plots/veg_topo_plots'
in_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'
dt = '*'

In [ ]:
# Recreate the depth arrays like so
test_fns = h.fn_list(in_dir, f'{basin}_*{dt}*{snowprop}*{reproj_type}*')
test_ds_list = [h.load(fn) for fn in test_fns]
# plot them
for ds in test_ds_list:
    title = ds.description
    ax = h.plot_one(ds, vmin=-1, vmax=1, cmap='RdYlBu', return_ax=True, figsize=(4,4), title=title)
    ax.set_facecolor('k')

In [ ]:
# Create the depth arrays by adding the array values together in pairs
new_depth_list = []
for i in range(0, len(test_ds_list), 2):
    ds1 = test_ds_list[i]
    ds2 = test_ds_list[i+1]
    new_depth = ds1 + ds2
    new_depth.attrs['description'] = f'{ds2.description.split(' - ')[0]} snow depth'
    new_depth_list.append(new_depth)

In [ ]:
for depth in new_depth_list:
    title = depth.description
    ax = h.plot_one(depth, vmin=0, vmax=2, cmap='Blues', return_ax=True, figsize=(4,4), title=title)
    ax.set_facecolor('k')

In [ ]:
# Bin by slope now
for depth in new_depth_list:
    vals_grouped = group_vals(arr=depth, var_bins_arr=slope_bins_arr)
    plot_boxplot(vals=vals_grouped, title=f'{depth.description} {dt}', ylab='snow depth [m]', ylims=(-0.1, 2.5))

# Plot model error (snow depth differences) by veg_height

In [ ]:
h.plot_one(veg_height)

In [ ]:
h.plot_hist(veg_height)

In [ ]:
import copy

In [ ]:
def bin_var(arr: xr.DataArray, basinname: str, bins: list,
            plot_on: bool = True,
            cmap: str = 'viridis', figsize: tuple = (4, 6), title: str = 'binned'
            ):
    """Bin input array based on pre-determined classes
    Parameters
    ----------
        arr: Array data.
        basinname: Name of the basin.
        bins: Input list of pre-determined classes
        plot_on: If True, plot the binned array. Defaults to True.
        cmap: Colormap to use for plotting. Defaults to 'viridis'.
        figsize: Size of the plot. Defaults to (4, 6).
        title: Title of the plot. Defaults to 'binned'.
    Returns
    ----------
        binned_da: Binned data.
    """
    title = f'{basinname} {title}'
    # Bin slope
    binned_da = copy.deepcopy(arr)

    # Extract numpy array from dataset to do reassignment
    arr_bin = binned_da.data
    for jdx, bin in enumerate(bins):
        if jdx < len(bins)-1:
            arr_bin[(arr_bin>=bin) & (arr_bin<bins[jdx+1])] = jdx
        else:
            arr_bin[(arr_bin>=bin)] = jdx+1
        print(jdx, bin)

    # Reassign array to dataset
    binned_da.data = arr_bin

    # Plot new classes
    if plot_on:
        h.plot_one(binned_da, cmap=cmap, title=title, figsize=figsize)

    return binned_da

In [ ]:
# binned_veg_height = bin_var(arr=veg_height, basinname=basin, bins=list(np.unique(veg_height.data)), plot_on=False)
binned_veg_height = bin_var(arr=veg_height, basinname=basin, bins=np.unique(veg_height.data), plot_on=False)
h.plot_one(binned_veg_height)
h.plot_hist(binned_veg_height)


In [ ]:
for diff_fn in diff_fn_list:
    # This works fine for iSnobal runs, needs handling for the original other types, try uniform reprojection now
    diff_vals_grouped = group_vals(arr_fn=diff_fn, var_bins_arr=binned_veg_height, v='veg_height', bins=np.arange(0, 10))
    plot_boxplot(vals=diff_vals_grouped, arr_fn=PurePath(diff_fn).stem, v='veg_height', rot=True,
                 bin_labels=['0.0-0.25', '0.25-0.75', '0.75-1.0', '1.0-2.0', '2.0-2.5', '2.5-3.0', '3.0-7.5', '7.5-17.5', '17.5-37.5', '≥37.5'])

In [ ]:
h.plot_one(topo.veg_type)

Great, lovely, but should really do by land cover rather than veg height. Try with just the veg type and then read in your other file for labels

# By Veg_type

In [ ]:
np.unique(veg_type)

In [ ]:
import pandas as pd
# read in the actual landfire parameter assignment file
params_csv = '/uufs/chpc.utah.edu/common/home/skiles-group3/LANDFIRE/landfire_veg_params_assigned.csv'
df = pd.read_csv(params_csv)
df.info()

In [ ]:
h.plot_one(topo.veg_type, cmap='tab20')

In [ ]:
veg_type = topo.veg_type.load()

In [ ]:
# Use landfire140 column to determine the CLASSNAME for actual unique veg_types
basin_veg_types = np.unique(veg_type)
basin_CLASSNAMES = df[df['landfire140'].isin(basin_veg_types)]['CLASSNAME'].values
# print(basin_CLASSNAMES)

# Create new land cover classes
new_classes = [['Water'],
               ['Crop'],
               ['Wetland'],
               ['Grass', 'Meadow', 'Hay', 'Wheat', 'Savanna', 'Greasewood'],
               ['Shrub', 'Sagebrush'],
               ['Deciduous'],
               ['Evergreen', 'Pine'],
               ['Mixed', 'Wood', 'Forest'],
               ['Subalpine'],
               ['Alpine'],
               ['Snow'],
               ['Barren', 'Sparse'],
               ['Rural'],
               ['Urban', 'Developed'],
               ]
new_veg_classes = copy.deepcopy(veg_type)
new_class_idx = []
veg_type_dict = dict()
# Based on the landfire140 codes in the veg_type array, assign new classes based on the CLASSNAME column in the landfire_veg_params_assigned.csv file
for i, veg_type_code in enumerate(basin_veg_types):
    class_name = df[df['landfire140'] == veg_type_code]['CLASSNAME'].values[0]
    for j, new_class in enumerate(new_classes):
        if any(substring in class_name for substring in new_class):
            print(f'{i}: code {veg_type_code} [ {class_name} ] assigning -- {new_class}, {j}')
            new_veg_classes = new_veg_classes.where(veg_type != veg_type_code, other=j)
            veg_type_dict[veg_type_code] = j
            new_class_idx.append(j)
            # This ensures the first class identified is kept
            break

    # Create a catch-all value for any codes that don't match the new classes, assign them to a new class index of 99
    if veg_type_code not in veg_type_dict:
        print(f'Veg type code {veg_type_code} [ {class_name} ] assigning -- Unclassified, 99')
        new_veg_classes = new_veg_classes.where(veg_type != veg_type_code, other=99)
        veg_type_dict[veg_type_code] = 99

new_class_idx = np.unique(new_class_idx)

In [ ]:
# Now using the veg_type_dict, create a new array and assign new values based off the veg_type array
new_veg_type = copy.deepcopy(veg_type)

# Print out the key value pairs in the veg_type_dict
for k, v in veg_type_dict.items():
    print(f'Veg type code {k} assigned to new class index {v}')
    new_veg_type = new_veg_type.where(new_veg_type != k, other=v)

In [ ]:
new_class_idx

In [ ]:
binned_veg_type = bin_var(arr=new_veg_type, basinname=basin, bins=new_class_idx, plot_on=False)
h.plot_one(binned_veg_type)
h.plot_hist(binned_veg_type)


In [ ]:
new_class_idx

In [ ]:
# Redo new_classes so it only keeps the first class in the list
short_classes = [c[0] for c in new_classes]
short_classes

In [ ]:
# Based on the new class labels and the indices of the actual array, get the bin labels
bin_labels = np.array(short_classes)[new_class_idx]
bin_labels

In [ ]:
for diff_fn in diff_fn_list:
    # This works fine for iSnobal runs, needs handling for the original other types, try uniform reprojection now
    # diff_vals_grouped = group_vals(arr_fn=diff_fn, var_bins_arr=binned_veg_type, v='veg_type', bins=new_class_idx)
    diff_vals_grouped = group_vals(arr_fn=diff_fn, var_bins_arr=new_veg_type, v='veg_type', bins=new_class_idx)
    plot_boxplot(vals=diff_vals_grouped, arr_fn=PurePath(diff_fn).stem, v='veg_type', rot=True, ylims=(-1,2),
                 med_on=True, med_shift=0.4, med_xshift=0.1,
                 bin_labels=bin_labels)

In [ ]:
# Great, now group slope values by these land cover classes
slope_vals_grouped = group_vals(arr=slope, var_bins_arr=new_veg_type, v='veg_type', bins=new_class_idx)

plot_boxplot(vals=slope_vals_grouped, title='slope distribution', v='veg_type', rot=True, ylims=(0,40),
                med_on=False, med_shift=10, med_xshift=0.1, ylab='Slope [˚]', units='˚',
                bin_labels=bin_labels)

In [ ]:
elev = topo['dem'].load()

In [ ]:
# Great, now group slope values by these land cover classes
elev_vals_grouped = group_vals(arr=elev, var_bins_arr=new_veg_type, v='veg_type', bins=new_class_idx)

plot_boxplot(vals=elev_vals_grouped, title='elevation distribution', v='veg_type', rot=True, ylims=(1500,4500),
                med_on=False, med_shift=-750, med_xshift=0.25, ylab='Elevation [m]', units='m',
                bin_labels=bin_labels)